# Deep Learning Based Game Playing Agent Using Deep Q-Learning
**Deep Learning Techniques (DLT) Project | Tic-Tac-Toe DQN Agent**

* **Author**: Dinesh Moorthy
* **Framework**: Python 3.10+, TensorFlow / Keras 3.x, NumPy, Matplotlib
* **Core Algorithm**: Deep Q-Network (DQN) with Experience Replay and Valid-Action Masking

---
### Project Objective
This notebook demonstrates how an artificial intelligence agent learns an optimal game-playing strategy for **Tic-Tac-Toe** through **Deep Reinforcement Learning (DRL)**. 

Rather than using hardcoded rules or a static dataset, the agent generates its own training data by interacting with the environment, archiving state-action transitions into an **Experience Replay Buffer**, and optimizing a **Multi-Layer Perceptron (MLP)** via backpropagation to approximate the optimal action-value function $Q^*(s, a)$.


## 1. System Setup & Library Imports
Import TensorFlow, NumPy, Matplotlib, and verify hardware acceleration.


In [ ]:
import os
import random
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set random seeds for reproducibility
def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)

print(f"TensorFlow Version : {tf.__version__}")
print(f"GPU Available      : {tf.config.list_physical_devices('GPU')}")


## 2. Tic-Tac-Toe Environment Implementation
The environment manages the $3 \times 3$ grid, enforces move legality, detects terminal states (Win, Loss, Draw), and calculates scalar reinforcement rewards.

### State Representation:
* `+1` : AI Agent (X or current player)
* `-1` : Opponent (O)
* ` 0` : Empty Cell

### Reward Scheme:
* **Win**: `+1.0`
* **Loss**: `-1.0`
* **Draw**: `+0.2` (incentivizes defensive survival)
* **Step**: `0.0`


In [ ]:
class TicTacToeEnvironment:
    """
    Simulates a 3x3 discrete Tic-Tac-Toe game environment.
    """
    WIN_COMBINATIONS = [
        (0, 1, 2), (3, 4, 5), (6, 7, 8),  # Rows
        (0, 3, 6), (1, 4, 7), (2, 5, 8),  # Columns
        (0, 4, 8), (2, 4, 6)               # Diagonals
    ]

    def __init__(self):
        self.board = np.zeros(9, dtype=np.int8)
        self.reset()

    def reset(self):
        self.board = np.zeros(9, dtype=np.int8)
        return self.get_state()

    def get_state(self):
        return np.copy(self.board)

    def get_valid_actions(self):
        return [i for i in range(9) if self.board[i] == 0]

    def is_valid_action(self, action):
        return 0 <= action < 9 and self.board[action] == 0

    def step(self, action, player=1):
        if not self.is_valid_action(action):
            raise ValueError(f"Invalid action {action} attempted.")

        self.board[action] = player
        winner = self.check_winner()

        if winner is not None:
            if winner == 1:
                return self.get_state(), 1.0, True, {"winner": 1, "status": "win"}
            elif winner == -1:
                return self.get_state(), -1.0, True, {"winner": -1, "status": "loss"}
            elif winner == 0:
                return self.get_state(), 0.2, True, {"winner": 0, "status": "draw"}

        return self.get_state(), 0.0, False, {"winner": None, "status": "ongoing"}

    def check_winner(self):
        for c1, c2, c3 in self.WIN_COMBINATIONS:
            line_sum = self.board[c1] + self.board[c2] + self.board[c3]
            if line_sum == 3:
                return 1
            if line_sum == -3:
                return -1

        if len(self.get_valid_actions()) == 0:
            return 0  # Draw
        return None

    def is_terminal(self):
        return self.check_winner() is not None

    def render_cli(self):
        symbols = {1: "X", -1: "O", 0: " "}
        c = [symbols[v] for v in self.board]
        return f"\n {c[0]} | {c[1]} | {c[2]} \n---+---+---\n {c[3]} | {c[4]} | {c[5]} \n---+---+---\n {c[6]} | {c[7]} | {c[8]} \n"

# Quick test
env = TicTacToeEnvironment()
print("Initialized Board Preview:")
print(env.render_cli())


## 3. Deep Q-Network (DQN) Model Architecture
The Deep Q-Network is a **Multi-Layer Perceptron (MLP)** that maps the 9-element spatial board representation to continuous Q-value predictions for each of the 9 board actions.

* **Input Layer**: 9 neurons (one per board cell)
* **Hidden Layer 1**: 64 neurons, ReLU activation
* **Hidden Layer 2**: 64 neurons, ReLU activation
* **Output Layer**: 9 neurons, Linear activation ($Q(s, a_0)$ to $Q(s, a_8)$)
* **Loss Function**: Huber Loss (robust to outlier TD errors)
* **Optimizer**: Adam ($	ext{lr} = 0.001$)


In [ ]:
def build_dqn_model(input_dim=9, output_dim=9, hidden_units=(64, 64), learning_rate=0.001):
    """
    Constructs and compiles the Deep Q-Network Multi-Layer Perceptron.
    """
    inputs = layers.Input(shape=(input_dim,), name="board_state_input")
    x = layers.Dense(hidden_units[0], activation="relu", name="dense_hidden_1")(inputs)
    x = layers.Dense(hidden_units[1], activation="relu", name="dense_hidden_2")(x)
    outputs = layers.Dense(output_dim, activation="linear", name="q_value_output")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="tic_tac_toe_dqn")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.Huber()
    )
    return model

model = build_dqn_model()
model.summary()


## 4. DQN Agent with Experience Replay & Action Masking
The agent implements:
1. **$\epsilon$-Greedy Policy**: Balances exploration ($\epsilon$) and exploitation ($1 - \epsilon$) with strict legal-action filtering.
2. **Experience Replay Buffer**: Stores $(s, a, r, s', 	ext{done})$ tuples to decorrelate consecutive game states.
3. **Bellman Optimality Target Calculation with Masking**:
   $$y_i = \begin{cases} r_i & \text{if done} \\ r_i + \gamma \max_{a' \in \mathcal{A}_{\text{valid}}(s'_i)} Q(s'_i, a') & \text{otherwise} \end{cases}$$
4. **Target Clipping**: Bounding targets to $[-1.0, 1.0]$ prevents gradient and Q-value explosion.


In [ ]:
class DQNAgent:
    def __init__(
        self,
        state_dim=9,
        action_dim=9,
        learning_rate=0.001,
        gamma=0.95,
        epsilon=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.9995,
        memory_size=50000,
        batch_size=64,
    ):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size

        self.memory = deque(maxlen=memory_size)
        self.model = build_dqn_model(
            input_dim=state_dim,
            output_dim=action_dim,
            hidden_units=(64, 64),
            learning_rate=learning_rate
        )

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state.astype(np.float32), action, reward, next_state.astype(np.float32), done))

    def act(self, state, valid_actions, training=True):
        if not valid_actions:
            raise ValueError("No valid actions available.")

        # Exploration: random valid move
        if training and np.random.rand() < self.epsilon:
            return random.choice(valid_actions)

        # Exploitation: argmax Q among valid moves
        q_values = self.get_q_values(state)
        masked_q = {a: q_values[a] for a in valid_actions}
        return max(masked_q, key=masked_q.get)

    def get_q_values(self, state):
        state_tensor = tf.convert_to_tensor(state.astype(np.float32).reshape(1, 9))
        q_preds = self.model(state_tensor, training=False).numpy()
        return q_preds[0]

    def replay(self, batch_size=None):
        b_size = batch_size or self.batch_size
        if len(self.memory) < b_size:
            return 0.0

        minibatch = random.sample(self.memory, b_size)

        states = np.array([t[0] for t in minibatch], dtype=np.float32)
        actions = np.array([t[1] for t in minibatch], dtype=np.int32)
        rewards = np.array([t[2] for t in minibatch], dtype=np.float32)
        next_states = np.array([t[3] for t in minibatch], dtype=np.float32)
        dones = np.array([t[4] for t in minibatch], dtype=bool)

        states_tensor = tf.convert_to_tensor(states)
        next_states_tensor = tf.convert_to_tensor(next_states)

        current_q_targets = self.model(states_tensor, training=False).numpy()
        next_q_values = self.model(next_states_tensor, training=False).numpy()

        for i in range(b_size):
            if dones[i]:
                target = rewards[i]
            else:
                valid_next = [idx for idx in range(9) if next_states[i, idx] == 0]
                if valid_next:
                    max_next_q = np.max([next_q_values[i, act] for act in valid_next])
                else:
                    max_next_q = 0.0
                target = rewards[i] + self.gamma * max_next_q

            target = np.clip(target, -1.0, 1.0)
            current_q_targets[i, actions[i]] = target

        loss = float(self.model.train_on_batch(states, current_q_targets))
        return loss

    def decay_epsilon(self):
        if self.epsilon > self.epsilon_min:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def save(self, filepath="models/tic_tac_toe_dqn.keras"):
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        self.model.save(filepath)
        print(f"Model saved to: {filepath}")

    def load(self, filepath="models/tic_tac_toe_dqn.keras"):
        self.model = keras.models.load_model(filepath)
        print(f"Model loaded from: {filepath}")


## 5. Opponent Benchmark Strategies
To ensure comprehensive generalization, the agent is trained and evaluated against:
1. **Random Opponent**: Selects uniformly from available empty cells.
2. **Strategic Heuristic Opponent**:
   * Takes an immediate winning move if available.
   * Blocks an immediate opponent win if detected.
   * Prefers center/corner positions.


In [ ]:
class RandomOpponent:
    @staticmethod
    def get_action(env):
        return random.choice(env.get_valid_actions())


class StrategicOpponent:
    @staticmethod
    def get_action(env):
        valid_actions = env.get_valid_actions()
        board = env.board

        # 1. Take immediate win
        for action in valid_actions:
            board[action] = -1
            if env.check_winner() == -1:
                board[action] = 0
                return action
            board[action] = 0

        # 2. Block opponent win
        for action in valid_actions:
            board[action] = 1
            if env.check_winner() == 1:
                board[action] = 0
                return action
            board[action] = 0

        # 3. Prefer center
        if 4 in valid_actions:
            return 4

        # 4. Prefer corners
        corners = [a for a in [0, 2, 6, 8] if a in valid_actions]
        if corners:
            return random.choice(corners)

        return random.choice(valid_actions)


## 6. Training Pipeline (10,000 Episodes)
Execute the training loop. The agent encounters mixed opponents (50% Random, 50% Strategic) and alternates starting turns.


In [ ]:
def train_agent(episodes=10000, batch_size=64, lr=0.001, gamma=0.95, log_interval=1000):
    set_seed(42)
    env = TicTacToeEnvironment()
    agent = DQNAgent(
        learning_rate=lr,
        gamma=gamma,
        epsilon=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.9995,
        batch_size=batch_size
    )

    random_opp = RandomOpponent()
    strategic_opp = StrategicOpponent()

    recent_outcomes = []
    losses_history = []
    win_rates, draw_rates, loss_rates, epsilons = [], [], [], []
    rewards_history, checkpoint_episodes = [], []

    print("=" * 65)
    print(f"STARTING DQN TRAINING: {episodes:,} EPISODES")
    print("=" * 65)

    for episode in range(1, episodes + 1):
        state = env.reset()
        episode_reward = 0.0
        episode_losses = []

        # 50% Random, 50% Strategic opponent
        opponent = strategic_opp if random.random() < 0.5 else random_opp

        # 50% AI first, 50% Opponent first
        if random.random() < 0.5:
            opp_act = opponent.get_action(env)
            state, _, done, _ = env.step(opp_act, player=-1)

        done = False
        while not done:
            valid_actions = env.get_valid_actions()
            if not valid_actions:
                break

            action = agent.act(state, valid_actions, training=True)
            next_state, reward, done, info = env.step(action, player=1)
            episode_reward += reward

            if done:
                agent.remember(state, action, reward, next_state, True)
                loss = agent.replay()
                if loss > 0:
                    episode_losses.append(loss)
                recent_outcomes.append(info.get("status", "unknown"))
                break

            opp_act = opponent.get_action(env)
            opp_next_state, opp_reward, opp_done, opp_info = env.step(opp_act, player=-1)

            if opp_done:
                final_ai_reward = -1.0 if opp_info.get("winner") == -1 else 0.2
                episode_reward += final_ai_reward
                agent.remember(state, action, final_ai_reward, opp_next_state, True)
                loss = agent.replay()
                if loss > 0:
                    episode_losses.append(loss)
                recent_outcomes.append(opp_info.get("status", "unknown"))
                done = True
            else:
                agent.remember(state, action, 0.0, opp_next_state, False)
                loss = agent.replay()
                if loss > 0:
                    episode_losses.append(loss)
                state = opp_next_state

        agent.decay_epsilon()
        rewards_history.append(episode_reward)

        if episode_losses:
            losses_history.append(float(np.mean(episode_losses)))

        if episode % log_interval == 0:
            window = recent_outcomes[-log_interval:]
            w = window.count("win")
            d = window.count("draw")
            l = window.count("loss")
            tot = len(window)

            win_rate = (w / tot) * 100 if tot > 0 else 0.0
            draw_rate = (d / tot) * 100 if tot > 0 else 0.0
            loss_rate = (l / tot) * 100 if tot > 0 else 0.0
            avg_loss = float(np.mean(losses_history[-log_interval:])) if losses_history else 0.0

            checkpoint_episodes.append(episode)
            win_rates.append(win_rate)
            draw_rates.append(draw_rate)
            loss_rates.append(loss_rate)
            epsilons.append(agent.epsilon)

            print(f"Episode {episode:6d}/{episodes} | Epsilon: {agent.epsilon:.4f} | Win: {win_rate:5.1f}% | Draw: {draw_rate:5.1f}% | Loss: {loss_rate:5.1f}% | Avg Loss: {avg_loss:.5f}", flush=True)

    agent.save("models/tic_tac_toe_dqn.keras")
    print("=" * 65)
    print("TRAINING COMPLETED SUCCESSFULLY!")

    history = {
        "episodes": checkpoint_episodes,
        "win_rates": win_rates,
        "draw_rates": draw_rates,
        "loss_rates": loss_rates,
        "epsilons": epsilons,
        "losses": losses_history,
        "rewards": rewards_history
    }
    return agent, history

trained_agent, history = train_agent(episodes=10000, log_interval=1000)


## 7. Results & Analytical Visualizations
Generate metric progression graphs illustrating:
1. **Training Loss Convergence (Huber Loss)**
2. **Win / Draw / Loss Rates Progression**
3. **Epsilon Decay Schedule (Exploration vs. Exploitation)**
4. **Cumulative Reward per Episode**


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(15, 10))

# 1. Training Loss
axs[0, 0].plot(history["losses"], color="#1f77b4", alpha=0.6, label="Huber Loss")
if len(history["losses"]) >= 50:
    w = 50
    smooth = np.convolve(history["losses"], np.ones(w)/w, mode="valid")
    axs[0, 0].plot(range(w, len(history["losses"]) + 1), smooth, color="#d62728", label=f"Moving Avg ({w})")
axs[0, 0].set_title("DQN Training Loss Over Updates")
axs[0, 0].set_xlabel("Replay Update Step")
axs[0, 0].set_ylabel("Loss (Huber)")
axs[0, 0].grid(True, linestyle="--", alpha=0.6)
axs[0, 0].legend()

# 2. Win / Draw / Loss Rates
ep_pts = history["episodes"]
axs[0, 1].plot(ep_pts, history["win_rates"], color="#2ca02c", linewidth=2, label="Win Rate (%)")
axs[0, 1].plot(ep_pts, history["draw_rates"], color="#ff7f0e", linewidth=2, label="Draw Rate (%)")
axs[0, 1].plot(ep_pts, history["loss_rates"], color="#d62728", linewidth=2, label="Loss Rate (%)")
axs[0, 1].set_title("Agent Performance Progression Over Training")
axs[0, 1].set_xlabel("Episode")
axs[0, 1].set_ylabel("Percentage (%)")
axs[0, 1].grid(True, linestyle="--", alpha=0.6)
axs[0, 1].legend()

# 3. Epsilon Decay
axs[1, 0].plot(ep_pts, history["epsilons"], color="#9467bd", linewidth=2, label="Epsilon (ε)")
axs[1, 0].set_title("Epsilon Decay Schedule")
axs[1, 0].set_xlabel("Episode")
axs[1, 0].set_ylabel("Exploration Rate (ε)")
axs[1, 0].grid(True, linestyle="--", alpha=0.6)
axs[1, 0].legend()

# 4. Reward History
all_eps = range(1, len(history["rewards"]) + 1)
axs[1, 1].plot(all_eps, history["rewards"], color="#17becf", alpha=0.2, label="Raw Reward")
if len(history["rewards"]) >= 50:
    w = 50
    smooth_r = np.convolve(history["rewards"], np.ones(w)/w, mode="valid")
    axs[1, 1].plot(list(all_eps)[w-1:], smooth_r, color="#005580", label=f"Rolling Avg ({w})")
axs[1, 1].set_title("Cumulative Reward per Episode")
axs[1, 1].set_xlabel("Episode")
axs[1, 1].set_ylabel("Reward")
axs[1, 1].grid(True, linestyle="--", alpha=0.6)
axs[1, 1].legend()

plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/training_summary.png", dpi=300)
plt.show()


## 8. Empirical Benchmark Evaluation (4,000 Matches)
Benchmark the **Untrained Baseline** against the **Trained DQN Agent** across 1,000 matches per opponent with alternating turn order.


In [ ]:
def evaluate_agent(agent, opponent, num_games=1000):
    env = TicTacToeEnvironment()
    wins, draws, losses = 0, 0, 0

    for g in range(num_games):
        state = env.reset()
        if g % 2 == 1:
            opp_act = opponent.get_action(env)
            state, _, done, _ = env.step(opp_act, player=-1)

        done = False
        while not done:
            valid_actions = env.get_valid_actions()
            if not valid_actions:
                break

            action = agent.act(state, valid_actions, training=False)
            next_state, reward, done, info = env.step(action, player=1)

            if done:
                winner = info.get("winner")
                if winner == 1: wins += 1
                elif winner == 0: draws += 1
                else: losses += 1
                break

            opp_act = opponent.get_action(env)
            opp_next_state, _, opp_done, opp_info = env.step(opp_act, player=-1)

            if opp_done:
                winner = opp_info.get("winner")
                if winner == -1: losses += 1
                elif winner == 0: draws += 1
                else: wins += 1
                done = True
            else:
                state = opp_next_state

    return {
        "Total": num_games,
        "Wins": wins,
        "Draws": draws,
        "Losses": losses,
        "Win Rate (%)": f"{(wins/num_games)*100:.2f}%",
        "Draw Rate (%)": f"{(draws/num_games)*100:.2f}%",
        "Loss Rate (%)": f"{(losses/num_games)*100:.2f}%"
    }

untrained = DQNAgent(epsilon=1.0)
rand_opp = RandomOpponent()
strat_opp = StrategicOpponent()

print("Running 4,000 benchmark matches...")
benchmarks = {
    "Untrained vs Random"   : evaluate_agent(untrained, rand_opp, 1000),
    "Untrained vs Strategic": evaluate_agent(untrained, strat_opp, 1000),
    "Trained vs Random"     : evaluate_agent(trained_agent, rand_opp, 1000),
    "Trained vs Strategic"  : evaluate_agent(trained_agent, strat_opp, 1000)
}

import pandas as pd
df = pd.DataFrame.from_dict(benchmarks, orient="index")
print("\n" + "=" * 70)
print("BENCHMARK EVALUATION SUMMARY")
print("=" * 70)
display(df)


## 9. Interactive Play Against the AI in Colab
Play an interactive match inside Google Colab. The cell visualizes real-time **predicted Q-values for every board position** before each move!


In [ ]:
def play_in_notebook(agent, human_starts=True):
    env = TicTacToeEnvironment()
    human_sym = 1 if human_starts else -1
    ai_sym = -human_sym
    
    print("=" * 55)
    print(f"PLAYING AS: {'X (First)' if human_sym == 1 else 'O (Second)'}")
    print("Board Positions:")
    print(" 0 | 1 | 2 \n---+---+---\n 3 | 4 | 5 \n---+---+---\n 6 | 7 | 8 \n")
    print("=" * 55)

    if not human_starts:
        # AI moves first
        state = env.get_state()
        act = agent.act(state, env.get_valid_actions(), training=False)
        env.step(act, player=ai_sym)
        print(f"AI took cell: {act}")

    done = False
    while not done:
        print(env.render_cli())
        valid_actions = env.get_valid_actions()
        
        # Human input
        move = None
        while move is None:
            try:
                val = int(input(f"Enter your move {valid_actions}: "))
                if val in valid_actions:
                    move = val
                else:
                    print(f"Invalid move! Choose from {valid_actions}.")
            except ValueError:
                print("Enter a number 0-8.")

        _, _, done, info = env.step(move, player=human_sym)
        if done:
            print(env.render_cli())
            winner = info.get("winner")
            if winner == human_sym:
                print("VICTORY! You won!")
            elif winner == 0:
                print("DRAW! Stalemate reached.")
            else:
                print("AI WINS!")
            break

        # AI Turn
        state = env.get_state()
        ai_state = state if ai_sym == 1 else -state
        q_vals = agent.get_q_values(ai_state)
        valid_actions = env.get_valid_actions()

        print("\nAI Real-Time Predicted Q-Values:")
        for va in valid_actions:
            print(f"  Cell {va}: Q = {q_vals[va]:+6.3f}")

        ai_act = agent.act(ai_state, valid_actions, training=False)
        print(f"AI Chooses Cell: {ai_act}")
        _, _, done, info = env.step(ai_act, player=ai_sym)

        if done:
            print(env.render_cli())
            winner = info.get("winner")
            if winner == ai_sym:
                print("AI WINS! (DQN Optimal Decision)")
            elif winner == 0:
                print("DRAW! (Well defended)")
            else:
                print("VICTORY! You won!")
            break

# Run an interactive game (Uncomment to play):
play_in_notebook(trained_agent, human_starts=True)


## 10. Save & Download Trained Model
Export the trained `.keras` checkpoint to your local machine from Colab.


In [ ]:
try:
    from google.colab import files
    trained_agent.save("tic_tac_toe_dqn.keras")
    files.download("tic_tac_toe_dqn.keras")
    print("Downloading trained model checkpoint...")
except ImportError:
    print("Running locally. Model already saved to models/tic_tac_toe_dqn.keras")
